# CS-Jail-UR: Phase 2 (Exp 7 training + Exp 8 generation, GPU)

Runs headless with **Save Version → Save & Run All (Commit)**. Design: `docs/PHASE2_PLAN.md`;
driver: `scripts/phase2_kaggle.py`.

| `STAGE` | What it does | Needs | Time |
|---|---|---|---|
| `"smoke"` | DPO on 8 harmless pairs per model, then the adapter in vLLM (passed 2026-10-09) | nothing private | ~30–40 min |
| `"train_eval"` | This account's share of the adapters (C and B_ext × 3 seeds, C n-curve, D6 ablation) and Exp 8 generations, one queue per GPU | the private pair dataset | 1 account ~7 h, 2 ~3.5–4 h, 3 ~2.5–3 h |

**Splitting the work over Kaggle accounts (1–3).** Every account runs this same notebook with the same
`N_PARTS` (how many accounts take part) and its own `PART` (1, 2, 3). The split is fixed and printed at
the start (`phase2_kaggle.py parts`): Llama goes to the first queues, so with `N_PARTS = 2` part 2 and
with `N_PARTS = 3` part 3 run **Phi-3 only** and need no Hugging Face token. Parts share nothing while
running; send every part's `phase2_outputs_partXofN.zip` back for judging.

**Time guard and resume.** Kaggle kills a run at 12 h and may then lose its output, so at `STOP_AFTER_H`
the notebook stops its jobs, packages everything and ends normally (`TIME GUARD` in the log). To finish,
run the same part again with that version's output added as input (Add Input → Your Work → this
notebook): finished adapters and evaluations are restored and skipped.

Before saving, on every account:
1. Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account).
2. Add Input → the **private** pair dataset made from `outputs/phase2_upload.zip` (harmful text: share
   it only with the team's accounts, or upload the zip as that account's own PRIVATE dataset). Any
   dataset name works: the notebook searches `/kaggle/input` for it.
3. Parts that run Llama: Add-ons → Secrets → **HF_TOKEN** from a Hugging Face account that has accepted
   the Llama-3.2 licence. The notebook stops within minutes if it is missing.
4. Set `N_PARTS` and `PART` below.

Nothing is judged here (Kaggle cannot reach the signed-in Ollama judge). Download
`phase2_outputs_partXofN.zip` (and `phase2_adapters_partXofN.zip` if wanted) from the version's
**Output** tab; the laptop judges them. Keep this notebook **private**.


In [ ]:
# Set before "Save & Run All".
STAGE = "train_eval"                     # "train_eval" (the smoke test passed on 2026-10-09)
N_PARTS = 1                              # how many Kaggle accounts share the work: 1, 2 or 3 (same on every account)
PART = 1                                 # THIS account's share: 1 .. N_PARTS (a different one on each account)
EPOCHS = 4                               # fixed by the learning check of the first full run (2 failed, 4 passed)
MODELS = ["phi3", "llama32"]             # smoke stage only
KEEP_ADAPTERS = True                     # also zip the trained LoRA adapters (~100 MB each)
STOP_AFTER_H = 11.0                      # time guard: stop, package, end normally (Kaggle kills at 12 h, output lost)

import glob, json, os, signal, subprocess, time
T0 = time.time()
TAG = f"part{PART}of{N_PARTS}"
assert 1 <= PART <= N_PARTS <= 3, "PART must be 1..N_PARTS and N_PARTS at most 3"
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip results so far into /kaggle/working (overwrites): generations, manifests, gate records."""
    keep = [p for p in ("outputs/phase2_smoke", "outputs/phase2_gate", "outputs/exp8") if os.path.exists(p)]
    keep += glob.glob("outputs/models*/*/training_manifest.json")
    # capture_output: a child writing to the kernel's own stdout can block the notebook for good
    if keep:
        subprocess.run(f"rm -f /kaggle/working/phase2_outputs_{TAG}.zip && zip -qr /kaggle/working/phase2_outputs_{TAG}.zip "
                       + " ".join(keep) + " -x 'outputs/phase2_smoke/*/adapter/*.safetensors'", shell=True,
                       capture_output=True)
    if KEEP_ADAPTERS and os.path.isdir("outputs/models"):
        subprocess.run(f"rm -f /kaggle/working/phase2_adapters_{TAG}.zip && "
                       f"zip -qr /kaggle/working/phase2_adapters_{TAG}.zip outputs/models", shell=True, capture_output=True)


def log_tail(path, lines=2, max_bytes=20000):
    """The end of a log, short and printed by Python (visible in the cell output). Progress bars
    rewrite one line with carriage returns thousands of times, so split on those too."""
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - max_bytes))
            text = f.read().decode("utf-8", "replace")
    except OSError:
        return "(no log yet)"
    parts = [p.strip()[-300:] for p in text.replace("\r", "\n").split("\n") if p.strip()]
    return "\n".join(parts[-lines:])


In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
# The training stack (TRL, PEFT, bitsandbytes) on top of the pinned vLLM stack.
sh('python -m uv pip install --python /root/py312/bin/python -e "' + '.[train,dev]' + '"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# GPU, gated-model token (only parts that run Llama need it), frozen dataset, test suite.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: fine for a Phi-3-only part; a part that runs Llama stops at the split check", flush=True)
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")


In [ ]:
# GPU smoke (PROTOCOL §9): fp16 DPO on 8 harmless pairs, then the adapter in vLLM, per model.
if STAGE == "smoke":
    for m in MODELS:
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step train")
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step serve")
        package()
    print("SMOKE PASSED for", MODELS)


In [ ]:
# Exp 7 + Exp 8 generation for THIS account's share: one queue per GPU, resumable, packaged every 10 minutes.
if STAGE == "train_eval":
    # The private bundle, found anywhere under /kaggle/input; an earlier version's output attached as
    # input is restored too (resume). Then the split: stops here if this part runs Llama without a token.
    sh("python scripts/phase2_kaggle.py stage")
    sh(f"python scripts/phase2_kaggle.py parts --n-parts {N_PARTS} --part {PART}")

    class TimeUp(Exception):
        """The time guard fired: end normally so Kaggle saves this version's output."""

    logs = {g: f"/kaggle/working/phase2_{TAG}_gpu{g}.log" for g in (0, 1)}
    procs = {g: subprocess.Popen(["python", "scripts/phase2_kaggle.py", "queue", "--part", str(PART),
                                  "--n-parts", str(N_PARTS), "--gpu", str(g), "--epochs", str(EPOCHS)],
                                 env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)), stdout=open(logs[g], "w"),
                                 stderr=subprocess.STDOUT, start_new_session=True)   # own group: the guard stops vLLM too
             for g in (0, 1)}
    try:
        while any(p.poll() is None for p in procs.values()):
            time.sleep(600)
            if time.time() - T0 > STOP_AFTER_H * 3600:
                for p in procs.values():
                    if p.poll() is None:
                        os.killpg(p.pid, signal.SIGTERM)
                time.sleep(30)
                package()
                raise TimeUp
            package()
            for g in procs:
                print(f"--- {TAG} gpu {g} ({(time.time() - T0) / 3600:.1f} h) ---\n" + log_tail(logs[g]), flush=True)
        package()
        bad = {g: p.returncode for g, p in procs.items() if p.returncode}
        for g in bad:   # the real error is in the log, not in this cell's traceback
            print(f"=== {TAG} gpu {g}: last 80 log lines ===\n" + log_tail(logs[g], lines=80, max_bytes=200000),
                  flush=True)
        if bad:
            raise RuntimeError(f"{TAG} failed on GPU(s) {sorted(bad)}; logs above")
        print(f"{TAG} DONE", flush=True)
    except TimeUp:
        print(f"TIME GUARD: stopped after {STOP_AFTER_H} h so Kaggle keeps this version's output. To finish, run "
              "the same PART again with THIS version's output added as input; it continues where it stopped.",
              flush=True)


In [ ]:
package()
sh("ls -lh /kaggle/working/*.zip")
print(f"DONE: download phase2_outputs_{TAG}.zip (and phase2_adapters_{TAG}.zip) from this version's Output tab.")
